# Experiment 3 — Domain Features + Honest Split + Fixed Inference

**Changes from experiment-2**
1. 11 domain features added to `FEATURE_COLS` (queue_pressure R²=45%, network_pressure R²=42%, …)
2. NaN imputation (ffill within series + fillna(0)) before lag computation
3. `nominal_capacity` and `trend` dropped (r<0.06, zero signal)
4. `rolling_forecast` looks up real domain features from `val_input_df` during inference
5. Chronological train/val split kept from experiment-2
6. `L1Loss` instead of `HuberLoss` to align training with WAPE metric

**Sections**
1. [Config](#1-config)
2. [Imports & Reproducibility](#2-imports--reproducibility)
3. [Data Loading](#3-data-loading)
4. [Exploratory Data Analysis](#4-exploratory-data-analysis)
5. [Baselines & Evaluation Metrics](#5-baselines--evaluation-metrics)
6. [Feature Engineering](#6-feature-engineering)
7. [PyTorch Dataset & DataLoader](#7-pytorch-dataset--dataloader)
8. [Model Definition](#8-model-definition)
9. [Training Loop](#9-training-loop)
10. [Validation Inference](#10-validation-inference)
11. [Save & Submit](#11-save--submit)

---
## 1. Config

In [1]:
from pathlib import Path

DATA_DIR     = Path("/kaggle/input/datasets/xitij27/dlam-bonus-project-dataset/data")
BASELINE_DIR = Path("/kaggle/input/datasets/xitij27/baseline-code/baseline")
OUTPUT_DIR   = Path("/kaggle/working")

TRAIN_PATH      = DATA_DIR / "train.csv"
VAL_INPUT_PATH  = DATA_DIR / "validation_input.csv"
VAL_INDEX_PATH  = DATA_DIR / "forecast_index_validation.csv"
METADATA_PATH   = DATA_DIR / "metadata.json"
CHECKPOINT_PATH = OUTPUT_DIR / "best_checkpoint.pt"
VAL_PREDS_PATH  = OUTPUT_DIR / "validation_predictions.csv"

# ── Hyperparameters ───────────────────────────────────────────────────────────
SEED         = 42
LOOKBACK     = 168
HORIZON      = 24
BATCH_SIZE   = 256
N_EPOCHS     = 100
LR           = 1e-3
WEIGHT_DECAY = 1e-4
EVAL_HORIZON = 336

# ── Feature knobs ─────────────────────────────────────────────────────────────
LAGS    = [1, 2, 3, 6, 12, 24, 48, 168]
WINDOWS = [6, 24, 168]

# Domain features selected from data analysis (R² > 3%, available in val_input_df)
# Dropped: nominal_capacity (r=-0.004), trend (r=+0.052), staffing_forecast (R²=0.2%),
#          throughput_disruption_risk_forecast (R²=0.1%)
DOMAIN_FEAT_COLS = [
    'workload_intensity',                  # R²=28.7%
    'demand_forecast',                     # R²=7.3%
    'upstream_quality_forecast',           # R²=5.5%
    'promotion_intensity',                 # R²=4.2%
    'shock_risk',                          # R²=24.6%
    'maintenance_known',                   # R²=6.1%
    'unit_reliability_forecast',           # R²=3.5%
    'queue_pressure_forecast',             # R²=45.2%  ← strongest predictor
    'network_pressure_forecast',           # R²=42.5%  ← 2nd strongest
    'event_load_forecast',                 # R²=17.1%
    'service_irregularity_risk_forecast',  # R²=3.2%
]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for p in [TRAIN_PATH, VAL_INPUT_PATH, VAL_INDEX_PATH, BASELINE_DIR / "baselines.py"]:
    print(f"{'OK' if p.exists() else 'MISSING':8s} {p}")

OK       /kaggle/input/datasets/xitij27/dlam-bonus-project-dataset/data/train.csv
OK       /kaggle/input/datasets/xitij27/dlam-bonus-project-dataset/data/validation_input.csv
OK       /kaggle/input/datasets/xitij27/dlam-bonus-project-dataset/data/forecast_index_validation.csv
OK       /kaggle/input/datasets/xitij27/baseline-code/baseline/baselines.py


---
## 2. Imports & Reproducibility

In [2]:
import sys
import json
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

if str(BASELINE_DIR) not in sys.path:
    sys.path.insert(0, str(BASELINE_DIR))
from baselines import make_all_baselines

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Python  {sys.version.split()[0]}")
print(f"pandas  {pd.__version__}")
print(f"numpy   {np.__version__}")
print(f"torch   {torch.__version__}")
print(f"Device  {DEVICE}")
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    print(f"GPU     {torch.cuda.get_device_name(0)}")

Python  3.12.12
pandas  2.3.3
numpy   2.0.2
torch   2.10.0+cu128
Device  cuda
GPU     Tesla T4


---
## 3. Data Loading

In [3]:
train_df     = pd.read_csv(TRAIN_PATH,     parse_dates=["timestamp"])
val_input_df = pd.read_csv(VAL_INPUT_PATH, parse_dates=["timestamp"])
val_index_df = pd.read_csv(VAL_INDEX_PATH, parse_dates=["timestamp"])

metadata = {}
if METADATA_PATH.exists():
    with open(METADATA_PATH) as f:
        metadata = json.load(f)

series_ids = sorted(train_df["series_id"].unique())

print(f"train_df      {train_df.shape}")
print(f"val_input_df  {val_input_df.shape}")
print(f"val_index_df  {val_index_df.shape}")
print(f"Series count  {len(series_ids)}")
train_df.head(3)

train_df      (414720, 25)
val_input_df  (32256, 24)
val_index_df  (32256, 2)
Series count  96


,series_id,timestamp,hour_sin,hour_cos,dow_sin,dow_cos,is_weekend,trend,workload_intensity,demand_forecast,...,unit_reliability_forecast,queue_pressure_forecast,network_pressure_forecast,event_load_forecast,service_irregularity_risk_forecast,throughput_disruption_risk_forecast,nominal_capacity,zone_sin,zone_cos,target
0,unit_000,2023-01-01 00:00:00,0.000000,1.000000,-0.781831,0.62349,1.0,-1.731704,0.001633,0.027929,...,1.000000,1.768614,0.818472,0.160134,0.00000,0.063503,82.690278,0.0,1.0,2.989556
1,unit_000,2023-01-01 01:00:00,0.258819,0.965926,-0.781831,0.62349,1.0,-1.731010,0.004694,0.000000,...,0.887202,1.806212,1.816860,0.004240,0.08081,0.000000,82.690278,0.0,1.0,3.199527
2,unit_000,2023-01-01 02:00:00,0.500000,0.866025,-0.781831,0.62349,1.0,-1.730316,0.012069,0.110750,...,0.986113,NaN,2.596726,0.208973,0.00000,0.009573,82.690278,0.0,1.0,3.425670


---
## 4. Exploratory Data Analysis

In [ ]:
summary = (
    train_df.groupby("series_id")["target"]
    .agg(["count", "mean", "std", "min", "max"])
    .round(3)
)
print(summary.describe().round(2))
summary.head()

In [ ]:
N_PLOT = 4
fig, axes = plt.subplots(N_PLOT, 1, figsize=(14, 3 * N_PLOT))
for ax, sid in zip(axes, series_ids[:N_PLOT]):
    s = train_df[train_df["series_id"] == sid].sort_values("timestamp")
    ax.plot(s["timestamp"], s["target"], linewidth=0.8)
    ax.set_title(f"series_id = {sid}")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.tight_layout()
plt.show()

---
## 5. Baselines & Evaluation Metrics

In [4]:
def mae(y_true, y_pred):             return np.mean(np.abs(y_true - y_pred))
def mse(y_true, y_pred):             return np.mean((y_true - y_pred) ** 2)
def rmse(y_true, y_pred):            return np.sqrt(mse(y_true, y_pred))
def mape(y_true, y_pred, eps=1e-8):  return np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + eps))) * 100
def smape(y_true, y_pred, eps=1e-8): return np.mean(np.abs(y_true - y_pred) / ((np.abs(y_true) + np.abs(y_pred)) / 2 + eps)) * 100
def wape(y_true, y_pred, eps=1e-8):  return np.sum(np.abs(y_true - y_pred)) / (np.sum(np.abs(y_true)) + eps) * 100

def evaluate(y_true, y_pred, label=""):
    metrics = dict(MAE=mae(y_true,y_pred), MSE=mse(y_true,y_pred), RMSE=rmse(y_true,y_pred),
                   MAPE=mape(y_true,y_pred), sMAPE=smape(y_true,y_pred), WAPE=wape(y_true,y_pred))
    prefix = f"[{label}] " if label else ""
    for name, val in metrics.items():
        print(f"  {prefix}{name:6s} = {val:.4f}")
    return metrics

In [5]:
held_out_rows, held_out_index_rows = [], []
for sid, grp in train_df.groupby("series_id", sort=False):
    grp = grp.sort_values("timestamp")
    held_out = grp.tail(EVAL_HORIZON)
    held_out_rows.append(held_out)
    held_out_index_rows.append(held_out[["series_id", "timestamp"]])

held_out_gt    = pd.concat(held_out_rows,       ignore_index=True)
held_out_index = pd.concat(held_out_index_rows, ignore_index=True)

train_eval_df = train_df.merge(
    held_out_gt[["series_id", "timestamp"]],
    on=["series_id", "timestamp"], how="left", indicator=True
).query('_merge == "left_only"').drop(columns="_merge")

print(f"Full train rows   : {len(train_df):,}")
print(f"Eval context rows : {len(train_eval_df):,}")
print(f"Held-out GT rows  : {len(held_out_gt):,}")

Full train rows   : 414,720
Eval context rows : 382,464
Held-out GT rows  : 32,256


In [6]:
baseline_preds_eval = make_all_baselines(train_eval_df, held_out_index)

all_metrics = {}
for name, pred_df in baseline_preds_eval.items():
    merged = held_out_index.merge(
        pred_df, on=["series_id", "timestamp"], how="left"
    ).merge(
        held_out_gt[["series_id", "timestamp", "target"]],
        on=["series_id", "timestamp"], how="left"
    )
    y_true = merged["target"].to_numpy(dtype=float)
    y_pred = merged["prediction"].to_numpy(dtype=float)
    mask   = ~(np.isnan(y_true) | np.isnan(y_pred))
    print(f"\n{'='*40}")
    all_metrics[name] = evaluate(y_true[mask], y_pred[mask], label=name)

print("\n── WAPE summary (held-out train tail) ────")
for name, m in all_metrics.items():
    print(f"  {name:20s}  {m['WAPE']:.4f}")


  [naive_last_value] MAE    = 5.8214
  [naive_last_value] MSE    = 57.0340
  [naive_last_value] RMSE   = 7.5521
  [naive_last_value] MAPE   = 55.6490
  [naive_last_value] sMAPE  = 65.7320
  [naive_last_value] WAPE   = 54.5040

  [lag24_repeat] MAE    = 4.4646
  [lag24_repeat] MSE    = 34.6364
  [lag24_repeat] RMSE   = 5.8853
  [lag24_repeat] MAPE   = 48.5955
  [lag24_repeat] sMAPE  = 49.4996
  [lag24_repeat] WAPE   = 41.8006

  [lag168_repeat] MAE    = 4.8584
  [lag168_repeat] MSE    = 40.4624
  [lag168_repeat] RMSE   = 6.3610
  [lag168_repeat] MAPE   = 54.0731
  [lag168_repeat] sMAPE  = 54.4251
  [lag168_repeat] WAPE   = 45.4876

  [seasonal_mean] MAE    = 3.3541
  [seasonal_mean] MSE    = 20.6407
  [seasonal_mean] RMSE   = 4.5432
  [seasonal_mean] MAPE   = 42.8468
  [seasonal_mean] sMAPE  = 34.6992
  [seasonal_mean] WAPE   = 31.4038

── WAPE summary (held-out train tail) ────
  naive_last_value      54.5040
  lag24_repeat          41.8006
  lag168_repeat         45.4876
  seasonal_m

---
## 6. Feature Engineering

**Changes from experiment-2:**
- `DOMAIN_FEAT_COLS` (11 features) added to `FEATURE_COLS`
- NaN imputation (ffill within series + fillna 0) applied before lag computation so gaps don't propagate
- `nominal_capacity` and `trend` dropped (zero/near-zero correlation with target)

In [7]:
def add_time_features(df, time_col="timestamp"):
    df = df.copy()
    t = pd.to_datetime(df[time_col])
    df["hour_sin"]   = np.sin(2 * np.pi * t.dt.hour / 24)
    df["hour_cos"]   = np.cos(2 * np.pi * t.dt.hour / 24)
    df["dow_sin"]    = np.sin(2 * np.pi * t.dt.dayofweek / 7)
    df["dow_cos"]    = np.cos(2 * np.pi * t.dt.dayofweek / 7)
    df["doy_sin"]    = np.sin(2 * np.pi * t.dt.dayofyear / 365)
    df["doy_cos"]    = np.cos(2 * np.pi * t.dt.dayofyear / 365)
    df["is_weekend"] = (t.dt.dayofweek >= 5).astype(float)
    return df


def impute_domain_features(df, domain_cols):
    """Forward-fill short gaps within each series, then zero-fill any remaining."""
    df = df.copy().sort_values(["series_id", "timestamp"])
    for col in domain_cols:
        if col in df.columns:
            df[col] = (
                df.groupby("series_id")[col]
                .transform(lambda x: x.ffill().fillna(0))
            )
    return df


def add_lag_features(df, lags, target_col="target"):
    df = df.copy().sort_values(["series_id", "timestamp"])
    for lag in lags:
        df[f"lag_{lag}"] = df.groupby("series_id")[target_col].shift(lag)
    return df


def add_rolling_features(df, windows, target_col="target"):
    df = df.copy().sort_values(["series_id", "timestamp"])
    for w in windows:
        grp = df.groupby("series_id")[target_col]
        df[f"roll_mean_{w}"] = grp.transform(lambda x: x.shift(1).rolling(w, min_periods=1).mean())
        df[f"roll_std_{w}"]  = grp.transform(lambda x: x.shift(1).rolling(w, min_periods=1).std().fillna(0))
    return df


def normalise_target(df, norm_stats):
    df = df.merge(norm_stats.reset_index(), on="series_id", how="left")
    df["target_norm"] = (df["target"] - df["target_mean"]) / df["target_std"]
    return df


# ── Feature columns ───────────────────────────────────────────────────────────
TIME_COLS = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "doy_sin", "doy_cos", "is_weekend"]
LAG_COLS  = [f"lag_{l}" for l in LAGS]
ROLL_COLS = [f"roll_mean_{w}" for w in WINDOWS] + [f"roll_std_{w}" for w in WINDOWS]

FEATURE_COLS = TIME_COLS + DOMAIN_FEAT_COLS + LAG_COLS + ROLL_COLS
print(f"Feature columns ({len(FEATURE_COLS)}):")
print(f"  Time    : {len(TIME_COLS)}")
print(f"  Domain  : {len(DOMAIN_FEAT_COLS)}  ← new")
print(f"  Lags    : {len(LAG_COLS)}")
print(f"  Rolling : {len(ROLL_COLS)}")
print(f"  Total   : {len(FEATURE_COLS)}  (was 21 in experiments 1-2)")

Feature columns (32):
  Time    : 7
  Domain  : 11  ← new
  Lags    : 8
  Rolling : 6
  Total   : 32  (was 21 in experiments 1-2)


---
## 6b. Diagnostic — do the domain features carry out-of-sample signal?

Test the experiment-3 premise directly, before any more LSTM tuning. The training
val-loss curve is *teacher-forced* (real lags every step) and overfits from epoch 1
regardless of features — so it can't tell us whether the domain features help. The
competition metric is a 336h autoregressive rollout where those features are the only
reliable signal. The cleanest way to check if that signal is real:

If `queue_pressure_forecast` (claimed R²=45%) and friends genuinely predict the target
out-of-sample, a **trivial per-hour regression** on the 11 features — no autoregression,
no LSTM — trained on the first ~166 days should beat `seasonal_mean` (WAPE 31.40) on the
held-out last 14 days.

- **Beats seasonal_mean** → signal is real; the LSTM is failing to use it (likely
  because inputs aren't normalised — raw lags ≈ 9–11 swamp domain features ≈ 0.01–2).
  Next step: normalise all features + select the checkpoint on rollout WAPE.
- **Doesn't beat it** → the data-analysis R² values are an in-sample / pooled artifact,
  and the whole experiment-3 strategy needs rethinking.

In [8]:
# ── Diagnostic A: do the domain features predict target OUT-OF-SAMPLE? ─────────
# Univariate fit quality on the held-out tail (last EVAL_HORIZON h/series) that no
# model below is trained on. "pooled" = across all series at once; "within" = mean
# of per-series r². Compares to the R² claimed in data-analysis.ipynb / README.
_imp = impute_domain_features(train_df, DOMAIN_FEAT_COLS)
_te  = pd.concat(
    [g.sort_values("timestamp").iloc[-EVAL_HORIZON:]
     for _, g in _imp.groupby("series_id", sort=False)],
    ignore_index=True,
)

claimed = {
    'queue_pressure_forecast': 45.2, 'network_pressure_forecast': 42.5,
    'workload_intensity': 28.7, 'shock_risk': 24.6, 'event_load_forecast': 17.1,
    'demand_forecast': 7.3, 'maintenance_known': 6.1, 'upstream_quality_forecast': 5.5,
    'promotion_intensity': 4.2, 'unit_reliability_forecast': 3.5,
    'service_irregularity_risk_forecast': 3.2,
}
_y = _te["target"].to_numpy(float)

def _r2_pooled(col):
    x = _te[col].to_numpy(float)
    return 100 * np.corrcoef(x, _y)[0, 1] ** 2 if np.std(x) > 0 else 0.0

def _r2_within(col):
    rs = []
    for _, g in _te.groupby("series_id", sort=False):
        x, y = g[col].to_numpy(float), g["target"].to_numpy(float)
        if np.std(x) > 0 and np.std(y) > 0:
            rs.append(np.corrcoef(x, y)[0, 1] ** 2)
    return 100 * np.mean(rs) if rs else 0.0

print(f"Out-of-sample r² on held-out tail ({len(_te):,} rows)\n")
print(f"{'feature':38s} {'pooled':>8s} {'within':>8s} {'claimed':>9s}")
print("-" * 67)
for col in sorted(DOMAIN_FEAT_COLS, key=_r2_pooled, reverse=True):
    print(f"{col:38s} {_r2_pooled(col):7.1f}% {_r2_within(col):7.1f}% {claimed.get(col, float('nan')):8.1f}%")

Out-of-sample r² on held-out tail (32,256 rows)

feature                                  pooled   within   claimed
-------------------------------------------------------------------
queue_pressure_forecast                   46.8%    53.5%     45.2%
network_pressure_forecast                 42.5%    50.4%     42.5%
workload_intensity                        29.7%    37.8%     28.7%
shock_risk                                18.2%    23.6%     24.6%
event_load_forecast                       13.5%    14.2%     17.1%
maintenance_known                          8.6%    17.8%      6.1%
unit_reliability_forecast                  4.6%     4.8%      3.5%
promotion_intensity                        3.1%     5.6%      4.2%
demand_forecast                            3.0%     4.5%      7.3%
service_irregularity_risk_forecast         2.2%     3.6%      3.2%
upstream_quality_forecast                  1.1%     2.0%      5.5%


In [9]:
# ── Diagnostic B: can a trivial model on domain features beat seasonal_mean? ───
# Per-hour regression: predict target[t] from the domain features at t (exactly
# how val_input_df serves them at inference). Honest chronological split.
from sklearn.linear_model import LinearRegression

_imp = impute_domain_features(train_df, DOMAIN_FEAT_COLS)
_tr = pd.concat([g.sort_values("timestamp").iloc[:-EVAL_HORIZON]
                 for _, g in _imp.groupby("series_id", sort=False)], ignore_index=True)
_te = pd.concat([g.sort_values("timestamp").iloc[-EVAL_HORIZON:]
                 for _, g in _imp.groupby("series_id", sort=False)], ignore_index=True)
_tr, _te = add_time_features(_tr), add_time_features(_te)   # adds doy_sin/doy_cos
y_tr, y_te = _tr["target"].to_numpy(float), _te["target"].to_numpy(float)

def _report(name, pred):
    print(f"  {name:34s} WAPE = {wape(y_te, np.maximum(pred, 0.0)):6.3f}")

print(f"  {'seasonal_mean (target to beat)':34s} WAPE = {all_metrics['seasonal_mean']['WAPE']:6.3f}\n")

lr = LinearRegression().fit(_tr[DOMAIN_FEAT_COLS], y_tr)
_report("linear / domain-only", lr.predict(_te[DOMAIN_FEAT_COLS]))

_feat = DOMAIN_FEAT_COLS + TIME_COLS
lr2 = LinearRegression().fit(_tr[_feat], y_tr)
_report("linear / domain + time", lr2.predict(_te[_feat]))

try:
    import lightgbm as lgb
    gbm = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.05, num_leaves=31,
                            subsample=0.8, colsample_bytree=0.8,
                            random_state=SEED, n_jobs=-1, verbosity=-1)
    gbm.fit(_tr[_feat], y_tr)
    _report("lightgbm / domain + time", gbm.predict(_te[_feat]))
    print("\n  LightGBM gain importance (top 8):")
    for f, g in sorted(zip(_feat, gbm.booster_.feature_importance("gain")),
                       key=lambda t: -t[1])[:8]:
        print(f"    {f:34s} {g:12.0f}")
except Exception as e:
    print(f"  lightgbm skipped: {e}")

  seasonal_mean (target to beat)     WAPE = 31.404

  linear / domain-only               WAPE = 20.999
  linear / domain + time             WAPE = 21.121
  lightgbm / domain + time           WAPE = 21.201

  LightGBM gain importance (top 8):
    queue_pressure_forecast                40790071
    event_load_forecast                    15916396
    network_pressure_forecast              12842533
    shock_risk                              4424636
    maintenance_known                       2793117
    demand_forecast                         1224803
    workload_intensity                       884667
    upstream_quality_forecast                860366


In [10]:
# Per-series normalisation stats
norm_stats = (
    train_df.groupby("series_id")["target"]
    .agg(target_mean="mean", target_std="std")
)
norm_stats["target_std"] = norm_stats["target_std"].replace(0, 1.0)

# Build feature frame
# Order matters: impute BEFORE lags so NaNs don't propagate into lag columns
feat_df = add_time_features(train_df)
feat_df = impute_domain_features(feat_df, DOMAIN_FEAT_COLS)  # ffill + fillna(0)
feat_df = add_lag_features(feat_df, lags=LAGS)
feat_df = add_rolling_features(feat_df, windows=WINDOWS)
feat_df = normalise_target(feat_df, norm_stats)
feat_df = feat_df.dropna(subset=FEATURE_COLS)

print(f"Feature frame shape: {feat_df.shape}")
print(f"Rows dropped by dropna: {len(train_df) - len(feat_df):,} (lag_168 burn-in only)")
feat_df.head(3)

Feature frame shape: (398592, 44)
Rows dropped by dropna: 16,128 (lag_168 burn-in only)


,series_id,timestamp,hour_sin,hour_cos,dow_sin,dow_cos,is_weekend,trend,workload_intensity,demand_forecast,...,lag_168,roll_mean_6,roll_std_6,roll_mean_24,roll_std_24,roll_mean_168,roll_std_168,target_mean,target_std,target_norm
168,unit_000,2023-01-08 00:00:00,0.000000,1.000000,-0.781831,0.62349,1.0,-1.615124,0.001633,0.000000,...,2.989556,9.809046,2.574784,8.026581,2.231346,11.419906,6.986540,9.141582,5.083139,-0.862423
169,unit_000,2023-01-08 01:00:00,0.258819,0.965926,-0.781831,0.62349,1.0,-1.614430,0.004694,0.000000,...,3.199527,8.739943,3.160582,7.976535,2.292374,11.430431,6.975087,9.141582,5.083139,-0.750965
170,unit_000,2023-01-08 02:00:00,0.500000,0.866025,-0.781831,0.62349,1.0,-1.613736,0.012069,0.072132,...,3.425670,7.476135,2.633015,7.965633,2.304826,11.443079,6.961987,9.141582,5.083139,-0.871572


---
## 7. PyTorch Dataset & DataLoader

Chronological split: last 336 hours of each series held out for validation.
Train loader uses `shuffle=True` (safe — windows are pre-computed self-contained pairs).

In [11]:
class SlidingWindowDataset(Dataset):
    """
    X : (lookback, n_features)  float32
    y : (horizon,)              float32  normalised targets
    """
    def __init__(self, df, feature_cols, target_col="target_norm",
                 lookback=LOOKBACK, horizon=HORIZON):
        self.samples = []
        for _, grp in df.groupby("series_id", sort=False):
            grp   = grp.sort_values("timestamp")
            X_arr = grp[feature_cols].to_numpy(dtype=np.float32)
            y_arr = grp[target_col].to_numpy(dtype=np.float32)
            n     = len(grp)
            for start in range(0, n - lookback - horizon + 1):
                self.samples.append((
                    X_arr[start : start + lookback],
                    y_arr[start + lookback : start + lookback + horizon],
                ))

    def __len__(self):          return len(self.samples)
    def __getitem__(self, idx):
        X, y = self.samples[idx]
        return torch.from_numpy(X), torch.from_numpy(y)


# Chronological split — no temporal leakage
train_rows, val_rows = [], []
for sid, grp in feat_df.groupby("series_id", sort=False):
    grp = grp.sort_values("timestamp")
    train_rows.append(grp.iloc[:-EVAL_HORIZON])
    val_rows.append(grp.iloc[-EVAL_HORIZON:])

train_feat_df = pd.concat(train_rows, ignore_index=True)
val_feat_df   = pd.concat(val_rows,   ignore_index=True)

train_dataset = SlidingWindowDataset(train_feat_df, FEATURE_COLS)
val_dataset   = SlidingWindowDataset(val_feat_df,   FEATURE_COLS)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"Train windows : {len(train_dataset):,}")
print(f"Val windows   : {len(val_dataset):,}")
xb, yb = next(iter(train_loader))
print(f"Batch X : {tuple(xb.shape)}   Batch y : {tuple(yb.shape)}")

Train windows : 348,000
Val windows   : 13,920
Batch X : (256, 168, 32)   Batch y : (256, 24)


---
## 8. Model Definition

In [12]:
class MLPForecaster(nn.Module):
    def __init__(self, n_features, lookback=LOOKBACK, horizon=HORIZON, hidden=256, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(lookback * n_features, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden),                nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, horizon),
        )
    def forward(self, x): return self.net(x)


class LSTMForecaster(nn.Module):
    def __init__(self, n_features, horizon=HORIZON, hidden=128, n_layers=2, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, n_layers, batch_first=True,
                            dropout=dropout if n_layers > 1 else 0.0)
        self.head = nn.Linear(hidden, horizon)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1])


class TransformerForecaster(nn.Module):
    def __init__(self, n_features, horizon=HORIZON, d_model=64, nhead=4, n_layers=2, dim_ff=256, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.encoder    = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model, nhead, dim_ff, dropout, batch_first=True),
            num_layers=n_layers
        )
        self.head = nn.Linear(d_model, horizon)
    def forward(self, x):
        z = self.encoder(self.input_proj(x))
        return self.head(z[:, -1])

In [13]:
N_FEATURES = len(FEATURE_COLS)

model = LSTMForecaster(n_features=N_FEATURES, hidden=256, n_layers=2, dropout=0.1)
# model = MLPForecaster(n_features=N_FEATURES)
# model = TransformerForecaster(n_features=N_FEATURES)

model = model.to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTrainable parameters: {n_params:,}")
print(f"Input features      : {N_FEATURES}  (was 21 in experiments 1-2)")

LSTMForecaster(
  (lstm): LSTM(32, 256, num_layers=2, batch_first=True, dropout=0.1)
  (head): Linear(in_features=256, out_features=24, bias=True)
)

Trainable parameters: 829,464
Input features      : 32  (was 21 in experiments 1-2)


---
## 9. Training Loop

**Change from experiments 1-2:** `nn.L1Loss()` instead of `HuberLoss`.
WAPE is a weighted absolute error — L1 trains the model to minimise absolute errors directly.

In [ ]:
criterion = nn.L1Loss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)


def run_epoch(loader, model, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, n = 0.0, 0
    with torch.set_grad_enabled(training):
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            pred = model(X)
            loss = criterion(pred, y)
            if training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item() * len(X)
            n += len(X)
    return total_loss / n


history = {"train": [], "val": []}
best_val_loss = float("inf")

for epoch in range(1, N_EPOCHS + 1):
    train_loss = run_epoch(train_loader, model, criterion, optimizer)
    val_loss   = run_epoch(val_loader,   model, criterion)
    scheduler.step()
    history["train"].append(train_loss)
    history["val"].append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({"state_dict": model.state_dict(), "epoch": epoch}, CHECKPOINT_PATH)

    print(f"Epoch {epoch:3d}/{N_EPOCHS}  "
          f"train={train_loss:.5f}  val={val_loss:.5f}  "
          f"lr={scheduler.get_last_lr()[0]:.2e}" +
          ("  ← best" if val_loss == best_val_loss else ""))

print(f"\nBest val loss: {best_val_loss:.5f} → {CHECKPOINT_PATH}")

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(history["train"], label="train loss")
plt.plot(history["val"],   label="val loss")
plt.xlabel("epoch")
plt.ylabel("L1 loss (normalised)")
plt.title("Training curves")
plt.legend()
plt.tight_layout()
plt.show()

---
## 10. Validation Inference

**Key change from experiments 1-2:** `rolling_forecast` now accepts `future_domain_df`
(= `val_input_df`). For each 24-step rollout block, the real pre-provided domain
features (`queue_pressure_forecast`, `network_pressure_forecast`, etc.) are looked
up from `val_input_df` and written into the context rows instead of being left as
zeros or computed from predicted targets.

This stops error accumulation in the two strongest predictors (R²=45% and 42%) over
the 14 rollout blocks needed to cover the 336-hour horizon.

In [ ]:
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"])
model.eval()
print(f"Loaded checkpoint from epoch {ckpt['epoch']}")


def make_window(ctx, feature_cols, lookback=LOOKBACK):
    X = ctx.tail(lookback)[feature_cols].to_numpy(dtype=np.float32)
    if len(X) < lookback:
        X = np.vstack([np.zeros((lookback - len(X), X.shape[1]), dtype=np.float32), X])
    return X


@torch.no_grad()
def rolling_forecast(model, context_df, index_df, norm_stats, feature_cols,
                     future_domain_df=None,
                     lookback=LOOKBACK, horizon=HORIZON):
    """
    future_domain_df : optional DataFrame with real domain features for forecast
                       timestamps (val_input_df). When provided, domain feature
                       values for predicted rows are looked up here instead of
                       being left as zeros.
    """
    # Pre-index future domain features by (series_id, timestamp) for O(1) lookup
    future_lookup = {}
    if future_domain_df is not None:
        for sid, grp in future_domain_df.groupby("series_id", sort=False):
            future_lookup[sid] = grp.set_index("timestamp")

    all_preds = []
    for sid, idx_grp in index_df.groupby("series_id", sort=False):
        idx_grp    = idx_grp.sort_values("timestamp")
        timestamps = idx_grp["timestamp"].tolist()

        ctx = context_df[context_df["series_id"] == sid].sort_values("timestamp").copy()
        ctx = add_time_features(ctx)
        ctx = impute_domain_features(ctx, DOMAIN_FEAT_COLS)
        ctx = add_lag_features(ctx, LAGS)
        ctx = add_rolling_features(ctx, WINDOWS)
        ctx = normalise_target(ctx, norm_stats)
        ctx = ctx.dropna(subset=feature_cols)

        mean = float(norm_stats.loc[sid, "target_mean"])
        std  = float(norm_stats.loc[sid, "target_std"])

        t_idx = 0
        while t_idx < len(timestamps):
            X_t       = torch.from_numpy(make_window(ctx, feature_cols)).unsqueeze(0).to(DEVICE)
            pred_norm = model(X_t).squeeze(0).cpu().numpy()

            n_step      = min(horizon, len(timestamps) - t_idx)
            step_norm   = pred_norm[:n_step]
            step_denorm = np.maximum(step_norm * std + mean, 0.0)
            step_times  = timestamps[t_idx : t_idx + n_step]

            for ts, p in zip(step_times, step_denorm):
                all_preds.append({"series_id": sid, "timestamp": ts, "prediction": p})

            # Build context rows for the predicted block
            combined = list(ctx["target"].values) + list(step_denorm)
            new_rows = add_time_features(
                pd.DataFrame({"series_id": sid, "timestamp": step_times, "target": step_denorm})
            )

            # ── Look up real domain features from val_input_df ──────────────
            if sid in future_lookup:
                fut = future_lookup[sid]
                for col in DOMAIN_FEAT_COLS:
                    if col in fut.columns:
                        new_rows[col] = [
                            float(fut.loc[ts, col]) if ts in fut.index else 0.0
                            for ts in step_times
                        ]
            else:
                for col in DOMAIN_FEAT_COLS:
                    new_rows[col] = 0.0
            # ───────────────────────────────────────────────────────────────

            for lag in LAGS:
                new_rows[f"lag_{lag}"] = [
                    combined[len(ctx) + i - lag] if len(ctx) + i - lag >= 0 else np.nan
                    for i in range(n_step)
                ]
            for w in WINDOWS:
                new_rows[f"roll_mean_{w}"] = float(np.mean(combined[-w:]))
                new_rows[f"roll_std_{w}"]  = float(np.std(combined[-w:]))
            new_rows["target_mean"] = float(mean)
            new_rows["target_std"]  = float(std)
            new_rows["target_norm"] = step_norm
            new_rows = new_rows.fillna(0.0)
            ctx   = pd.concat([ctx, new_rows], ignore_index=True)
            t_idx += n_step

    return pd.DataFrame(all_preds)

In [ ]:
# Competition predictions — pass val_input_df so real domain features are used
predictions_df = rolling_forecast(
    model, train_df, val_index_df, norm_stats, FEATURE_COLS,
    future_domain_df=val_input_df,
)
print(f"Predictions: {len(predictions_df):,} rows  (expected {len(val_index_df):,})")
predictions_df.head()

In [ ]:
# Local evaluation on held-out train tail (no future_domain_df — oracle not available here)
eval_preds_df = rolling_forecast(
    model, train_eval_df, held_out_index, norm_stats, FEATURE_COLS,
    future_domain_df=None,
)

eval_merged = held_out_index.merge(
    eval_preds_df, on=["series_id", "timestamp"], how="left"
).merge(
    held_out_gt[["series_id", "timestamp", "target"]], on=["series_id", "timestamp"], how="left"
)
y_true_e = eval_merged["target"].to_numpy(dtype=float)
y_pred_e = eval_merged["prediction"].to_numpy(dtype=float)
mask_e   = ~(np.isnan(y_true_e) | np.isnan(y_pred_e))

print("── Model (held-out train tail) ───────────")
model_metrics = evaluate(y_true_e[mask_e], y_pred_e[mask_e], label="model")
print(f"\n── vs. seasonal_mean baseline ────────────")
print(f"  seasonal_mean WAPE = {all_metrics['seasonal_mean']['WAPE']:.4f}")
print(f"  model         WAPE = {model_metrics['WAPE']:.4f}")
delta = (all_metrics['seasonal_mean']['WAPE'] - model_metrics['WAPE']) / all_metrics['seasonal_mean']['WAPE'] * 100
print(f"  Improvement        = {delta:+.1f}%")

In [ ]:
VIS_SERIES = series_ids[0]

pred_s = predictions_df[predictions_df["series_id"] == VIS_SERIES].sort_values("timestamp")
base_s = baseline_preds_eval["seasonal_mean"]
base_s = base_s[base_s["series_id"] == VIS_SERIES].sort_values("timestamp")
hist_s = train_df[train_df["series_id"] == VIS_SERIES].sort_values("timestamp").tail(168)

plt.figure(figsize=(14, 4))
plt.plot(hist_s["timestamp"], hist_s["target"],     label="history",       color="gray",  linewidth=0.8)
plt.plot(pred_s["timestamp"], pred_s["prediction"], label="model",         color="green", linewidth=1.2)
plt.plot(base_s["timestamp"], base_s["prediction"], label="seasonal_mean", color="coral", linewidth=1.0, linestyle="--")
plt.legend()
plt.title(f"{VIS_SERIES} — experiment-3 model vs seasonal_mean")
plt.tight_layout()
plt.show()

---
## 11. Save & Submit

In [ ]:
predictions_df[["series_id", "timestamp", "prediction"]].to_csv(VAL_PREDS_PATH, index=False)
print(f"Predictions saved  → {VAL_PREDS_PATH}")
print(f"Checkpoint saved   → {CHECKPOINT_PATH}")
print("\nDownload both files from the Kaggle output panel to add to your submission.")

---
### Changes vs. Previous Experiments

| What | Exp-1 | Exp-2 | Exp-3 |
|---|---|---|---|
| Train/val split | `random_split` (leakage) | Chronological | Chronological |
| Feature count | 21 | 21 | **32** |
| Domain features | None | None | **11 added** |
| NaN imputation | None (dropna) | None (dropna) | **ffill + fillna(0)** |
| Dropped features | — | — | **nominal_capacity, trend** |
| Loss | HuberLoss | HuberLoss | **L1Loss** |
| Inference domain feats | Computed from lags | Computed from lags | **Looked up from val_input_df** |
| Model | LSTM 256h | LSTM 256h | LSTM 256h |

---
## 12. Direct regression submission  ← run this instead of Section 11

Diagnostic B showed a per-hour regression on the domain features scores **WAPE ≈ 21**
vs `seasonal_mean` 31.40. And `validation_input.csv` has **no `target` column** — so the
target-lags that justified the autoregressive rollout in Section 10 don't even exist at
inference time. The correct model maps the *known future covariates* (`queue_pressure_forecast`,
`network_pressure_forecast`, …, all provided per timestamp) straight to the target. No
lookback window, no 14-block rollout, no error accumulation.

This section: (1) compares a few direct regressors on the honest held-out split,
(2) refits the best on all of `train.csv`, (3) writes `validation_predictions.csv`.
It **supersedes** Sections 10–11 — those LSTM cells can be skipped.

In [14]:
# ── Honest-split comparison: pick the best direct regressor ───────────────────
# Features = domain forecasts + time + per-series mean. NO target-lags (unavailable
# in validation_input.csv anyway). series_mean gives each unit its own baseline level.
from sklearn.linear_model import LinearRegression
import lightgbm as lgb

def prep(df):
    df = add_time_features(impute_domain_features(df, DOMAIN_FEAT_COLS))
    sm = norm_stats["target_mean"].rename("series_mean").reset_index()
    return df.merge(sm, on="series_id", how="left")

_full = prep(train_df)
_tr = pd.concat([g.sort_values("timestamp").iloc[:-EVAL_HORIZON]
                 for _, g in _full.groupby("series_id", sort=False)], ignore_index=True)
_te = pd.concat([g.sort_values("timestamp").iloc[-EVAL_HORIZON:]
                 for _, g in _full.groupby("series_id", sort=False)], ignore_index=True)
ytr, yte = _tr["target"].to_numpy(float), _te["target"].to_numpy(float)

def make_lgbm():
    return lgb.LGBMRegressor(n_estimators=600, learning_rate=0.03, num_leaves=63,
                             subsample=0.8, colsample_bytree=0.8, min_child_samples=50,
                             reg_lambda=1.0, random_state=SEED, n_jobs=-1, verbosity=-1)

def evaluate_model(name, model_fn, feats):
    m = model_fn(); m.fit(_tr[feats], ytr)
    w = wape(yte, np.maximum(m.predict(_te[feats]), 0.0))
    print(f"  {name:38s} WAPE = {w:6.3f}")
    return {"name": name, "wape": w, "model_fn": model_fn, "feats": feats}

print(f"  {'seasonal_mean (baseline)':38s} WAPE = {all_metrics['seasonal_mean']['WAPE']:6.3f}\n")
cands = [
    evaluate_model("linear / domain",                  LinearRegression, DOMAIN_FEAT_COLS),
    evaluate_model("linear / domain + series_mean",    LinearRegression, DOMAIN_FEAT_COLS + ["series_mean"]),
    evaluate_model("lightgbm / domain + time + smean", make_lgbm,        DOMAIN_FEAT_COLS + TIME_COLS + ["series_mean"]),
]
best = min(cands, key=lambda c: c["wape"])
print(f"\n  → best: {best['name']}  (held-out WAPE {best['wape']:.3f})")

  seasonal_mean (baseline)               WAPE = 31.404

  linear / domain                        WAPE = 20.999
  linear / domain + series_mean          WAPE = 17.975
  lightgbm / domain + time + smean       WAPE = 16.532

  → best: lightgbm / domain + time + smean  (held-out WAPE 16.532)


In [15]:
# ── Refit best model on ALL of train.csv, predict validation_input.csv, save ──
feats = best["feats"]
final_model = best["model_fn"]()
final_model.fit(_full[feats], _full["target"].to_numpy(float))

val_prep = prep(val_input_df)                       # same imputation + features
val_prep["prediction"] = np.maximum(final_model.predict(val_prep[feats]), 0.0)

# Align exactly to the required forecast index
submission = val_index_df.merge(
    val_prep[["series_id", "timestamp", "prediction"]],
    on=["series_id", "timestamp"], how="left",
)
assert submission["prediction"].notna().all(), "some index rows got no prediction — check (series_id, timestamp) alignment"
assert len(submission) == len(val_index_df), "row count mismatch vs forecast index"

submission[["series_id", "timestamp", "prediction"]].to_csv(VAL_PREDS_PATH, index=False)
print(f"Submission written : {len(submission):,} rows → {VAL_PREDS_PATH}")
print(f"Model              : {best['name']}")
print(f"Held-out WAPE      : {best['wape']:.3f}  (vs seasonal_mean {all_metrics['seasonal_mean']['WAPE']:.3f})")
print(f"Leaderboard        : expect similar if distribution shift stays <0.25σ\n")
print(submission["prediction"].describe().round(3))

Submission written : 32,256 rows → /kaggle/working/validation_predictions.csv
Model              : lightgbm / domain + time + smean
Held-out WAPE      : 16.532  (vs seasonal_mean 31.404)
Leaderboard        : expect similar if distribution shift stays <0.25σ

count    32256.000
mean        11.003
std          5.050
min          1.786
25%          7.133
50%         10.258
75%         14.100
max         37.904
Name: prediction, dtype: float64
